In [0]:
-- Stored Procedure voor fact_prices
CREATE OR REPLACE PROCEDURE gold.load_fact_prices()
SQL SECURITY INVOKER
BEGIN
  INSERT INTO gold.fact_prices
  SELECT
    s.coin_id,
    s.price DOUBLE,
    s.price_timestamp,
    unix_timestamp(s.price_timestamp) AS timestamp_unix,
    s.date
  FROM silver.crypto_prices s
  LEFT ANTI JOIN gold.fact_prices g -- alleen unieke records toevoegen
  ON s.coin_id = g.coin_id
  AND s.price_timestamp = g.price_timestamp;
END;

-- Stored Procedure voor fact_ohlc
CREATE OR REPLACE PROCEDURE gold.load_fact_ohlc()
SQL SECURITY INVOKER 
BEGIN
  INSERT INTO gold.fact_ohlc
  WITH daily AS (
    SELECT
      coin_id,
      FIRST(price) AS open_price,
      MAX(price) AS high_price,
      MIN(price) AS low_price,
      LAST(price) AS close_price,
      date
    FROM silver.crypto_prices
    GROUP BY coin_id, date
  )
  SELECT d.*
  FROM daily d
  LEFT ANTI JOIN gold.fact_ohlc g -- alleen unique records toevoegen
  ON d.coin_id = g.coin_id
  AND d.date = g.date;
END;